In [24]:
import os

In [25]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAQ_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [26]:
import requests

url = "https://api.openaq.org/v3/locations"
headers = {
    "X-API-Key": api_key
}
params = {
    "limit": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    print("OpenAQ connection successful")
elif response.status_code == 401:
    print("Authentication failed: check the OpenAQ API key")
elif response.status_code == 429:
    print("Rate limit reached: try again later")
else:
    print("Request failed:", response.text[:200])

HTTP status: 200
OpenAQ connection successful


### Fetch every page of Delhi locations

The first request returned exactly 100 locations, which is the page limit. The metadata says there are more than 100 results.

We must request page 2, page 3, and so on until a page contains fewer than 100 locations. Without this loop, we would use only the first page and miss possible Delhi PM2.5 stations.

In [27]:
import time

all_delhi_locations = []
page = 1
page_size = 100

while True:
    page_params = {
        "bbox": "76.80,28.40,77.40,28.90",
        "parameters_id": 2,
        "limit": page_size,
        "page": page,
    }

    page_response = requests.get(
        url,
        headers=headers,
        params=page_params,
        timeout=30,
    )

    if page_response.status_code == 401:
        raise RuntimeError("Authentication failed: check the OpenAQ API key")

    if page_response.status_code == 429:
        print("Rate limit reached. Wait and run this cell again.")
        break

    if page_response.status_code != 200:
        raise RuntimeError(
            f"OpenAQ request failed with HTTP {page_response.status_code}"
        )

    page_locations = page_response.json()["results"]
    all_delhi_locations.extend(page_locations)

    print(f"Page {page}: {len(page_locations)} locations")

    if len(page_locations) < page_size:
        break

    page += 1
    time.sleep(1)

print("Total locations collected:", len(all_delhi_locations))

Page 1: 100 locations
Page 2: 2 locations
Total locations collected: 102


### Inspect a location and its sensors

A location is the monitoring station. A sensor is the specific measuring device inside that station.

OpenAQ's later measurement endpoints need the sensor ID, not only the location ID. We inspect one result first so we understand the exact response structure before requesting sensors for all 102 locations.

In [28]:
first_location = all_delhi_locations[0]

print("Location ID:", first_location.get("id"))
print("Location name:", first_location.get("name"))
print("Coordinates:", first_location.get("coordinates"))
print("Provider:", first_location.get("provider"))

Location ID: 13
Location name: Delhi Technological University, Delhi - CPCB
Coordinates: {'latitude': 28.744, 'longitude': 77.12}
Provider: {'id': 168, 'name': 'CPCB'}


### Request the sensors for one location

Now we ask OpenAQ which sensors belong to this location. We will later keep only the sensor whose parameter is PM2.5 and record its first and last reading dates.

In [29]:
location_id = first_location["id"]
sensors_url = f"https://api.openaq.org/v3/locations/{location_id}/sensors"

sensors_response = requests.get(
    sensors_url,
    headers=headers,
    timeout=30,
)

print("HTTP status:", sensors_response.status_code)

if sensors_response.status_code == 200:
    sensors_payload = sensors_response.json()
    sensors = sensors_payload["results"]

    print("Sensors found:", len(sensors))

    for sensor in sensors:
        parameter = sensor.get("parameter", {})
        print(
            "Sensor ID:",
            sensor.get("id"),
            "| Parameter:",
            parameter.get("name"),
            "| First reading:",
            sensor.get("datetimeFirst"),
            "| Last reading:",
            sensor.get("datetimeLast"),
        )
elif sensors_response.status_code == 401:
    print("Authentication failed: check the OpenAQ API key")
elif sensors_response.status_code == 429:
    print("Rate limit reached: wait before trying again")
else:
    print("Request failed:", sensors_response.text[:200])

HTTP status: 200
Sensors found: 3
Sensor ID: 13866 | Parameter: no2 | First reading: {'utc': '2016-11-02T19:00:00Z', 'local': '2016-11-03T00:30:00+05:30'} | Last reading: {'utc': '2018-02-22T04:00:00Z', 'local': '2018-02-22T09:30:00+05:30'}
Sensor ID: 13864 | Parameter: pm25 | First reading: {'utc': '2016-11-02T19:00:00Z', 'local': '2016-11-03T00:30:00+05:30'} | Last reading: {'utc': '2018-02-22T04:00:00Z', 'local': '2018-02-22T09:30:00+05:30'}
Sensor ID: 24 | Parameter: o3 | First reading: None | Last reading: None


### Collect the PM2.5 sensor for every location

The locations endpoint gives us monitoring stations, but each station can contain several sensors for different pollutants.

We keep only the sensor whose parameter is `pm25`. For each matching station, we record the station identity, coordinates, provider, sensor ID, and first and last reading dates.

The helper also handles common API failures:
- 401 means the key is invalid or missing, so we stop.
- 429 means rate limiting, so we wait and retry.

In [30]:
def fetch_sensors_with_retry(location_id, max_retries=3):
    sensors_url = f"https://api.openaq.org/v3/locations/{location_id}/sensors"

    for attempt in range(max_retries):
        response = requests.get(
            sensors_url,
            headers=headers,
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()["results"]

        if response.status_code == 401:
            raise RuntimeError(
                "Authentication failed: check the OpenAQ API key"
            )

        if response.status_code == 429:
            wait_seconds = 10 * (attempt + 1)
            print(
                f"Rate limit reached for location {location_id}. "
                f"Waiting {wait_seconds} seconds."
            )
            time.sleep(wait_seconds)
            continue

        raise RuntimeError(
            f"Request failed for location {location_id} "
            f"with HTTP {response.status_code}"
        )

    raise RuntimeError(
        f"Could not fetch sensors for location {location_id} "
        "after several retries"
    )


station_records = []

for index, location in enumerate(all_delhi_locations, start=1):
    location_id = location["id"]
    sensors = fetch_sensors_with_retry(location_id)

    coordinates = location.get("coordinates") or {}
    provider = location.get("provider")

    if isinstance(provider, dict):
        provider = provider.get("name")

    for sensor in sensors:
        parameter = sensor.get("parameter") or {}

        if parameter.get("name", "").lower() != "pm25":
            continue

        first_reading = sensor.get("datetimeFirst") or {}
        last_reading = sensor.get("datetimeLast") or {}

        station_records.append(
            {
                "location_id": location_id,
                "sensor_id": sensor.get("id"),
                "name": location.get("name"),
                "latitude": coordinates.get("latitude"),
                "longitude": coordinates.get("longitude"),
                "provider": provider,
                "first_reading": first_reading.get("local"),
                "last_reading": last_reading.get("local"),
            }
        )

    if index % 10 == 0 or index == len(all_delhi_locations):
        print(
            f"Processed {index}/{len(all_delhi_locations)} locations; "
            f"PM2.5 stations found: {len(station_records)}"
        )

    time.sleep(1)

print("Total PM2.5 station records:", len(station_records))

Processed 10/102 locations; PM2.5 stations found: 14
Processed 20/102 locations; PM2.5 stations found: 27
Processed 30/102 locations; PM2.5 stations found: 44
Processed 40/102 locations; PM2.5 stations found: 55
Processed 50/102 locations; PM2.5 stations found: 65
Processed 60/102 locations; PM2.5 stations found: 85
Processed 70/102 locations; PM2.5 stations found: 102
Processed 80/102 locations; PM2.5 stations found: 119
Processed 90/102 locations; PM2.5 stations found: 135
Processed 100/102 locations; PM2.5 stations found: 145
Processed 102/102 locations; PM2.5 stations found: 147
Total PM2.5 station records: 147


### PM2.5 station table

The collection loop has finished. We now convert the collected PM2.5 records into a table so we can inspect each station, its sensor, and its available history.

In [32]:
import pandas as pd

pm25_table = pd.DataFrame(station_records)

print("Rows in table:", len(pm25_table))
print("Unique locations:", pm25_table["location_id"].nunique())

display(
    pm25_table.sort_values(
        by=["location_id", "first_reading"]
    ).reset_index(drop=True)
)

Rows in table: 147
Unique locations: 102


,location_id,sensor_id,name,latitude,longitude,provider,first_reading,last_reading
0,13,13864,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000,CPCB,2016-11-03T00:30:00+05:30,2018-02-22T09:30:00+05:30
1,15,30,IGI Airport,28.560000,77.094000,CPCB,None,None
2,16,34,Civil Lines,28.678700,77.226200,CPCB,None,None
3,17,35,"R K Puram, Delhi - DPCC",28.563262,77.186937,CPCB,2016-02-05T20:25:00+05:30,2018-02-22T02:45:00+05:30
4,17,12234787,"R K Puram, Delhi - DPCC",28.563262,77.186937,CPCB,2025-02-19T01:45:00+05:30,2026-09-11T15:45:00+05:30
...,...,...,...,...,...,...,...,...
142,6254665,15554728,"Commonwealth Sports Complex, Delhi - DPCC",28.615828,77.271992,N/A,2026-02-27T22:30:00+05:30,2026-09-11T16:00:00+05:30
143,6254666,15554740,"IGNOU_Maidan Garhi, Delhi - DPCC",28.493624,77.201159,N/A,2026-02-27T22:30:00+05:30,2026-09-11T16:00:00+05:30
144,6257818,15578291,"Cantonment Area, Delhi - DPCC",28.594169,77.125100,N/A,2026-03-02T18:30:00+05:30,2026-09-11T14:30:00+05:30
145,6299494,15888918,"Prashant Garden, Khora - UPPCB",28.611190,77.342060,N/A,2026-04-07T18:15:00+05:30,2026-09-11T16:00:00+05:30


In [33]:
from pathlib import Path

raw_data_folder = Path("../data/raw")
raw_data_folder.mkdir(parents=True, exist_ok=True)

stations_file = raw_data_folder / "stations.csv"
pm25_table.to_csv(stations_file, index=False)

print("Saved station table to:", stations_file)
print("Rows saved:", len(pm25_table))
print("Columns saved:", list(pm25_table.columns))

Saved station table to: ../data/raw/stations.csv
Rows saved: 147
Columns saved: ['location_id', 'sensor_id', 'name', 'latitude', 'longitude', 'provider', 'first_reading', 'last_reading']


### Check station coverage by year

The station CSV tells us when each sensor started and ended reporting. We now calculate the available year span for every PM2.5 sensor.

This is an initial coverage check. A first reading in 2016 and a last reading in 2018 tells us the sensor spans those years, but it does not prove that every day or every hour in between is complete.

In [37]:
coverage_table = pm25_table.copy()

coverage_table["first_reading"] = pd.to_datetime(
    coverage_table["first_reading"],
    errors="coerce"
)

coverage_table["last_reading"] = pd.to_datetime(
    coverage_table["last_reading"],
    errors="coerce"
)

coverage_table["first_year"] = coverage_table["first_reading"].dt.year
coverage_table["last_year"] = coverage_table["last_reading"].dt.year

coverage_table["coverage_years"] = (
    coverage_table["last_year"] - coverage_table["first_year"] + 1
)

coverage_table = coverage_table[
    [
        "location_id",
        "sensor_id",
        "name",
        "first_reading",
        "last_reading",
        "first_year",
        "last_year",
        "coverage_years",
    ]
].sort_values(
    by=["coverage_years", "first_reading"],
    ascending=[False, True]
)

display(coverage_table.reset_index(drop=True))

,location_id,sensor_id,name,first_reading,last_reading,first_year,last_year,coverage_years
0,8118,23534,New Delhi,2016-11-10 00:30:00+05:30,2026-09-15 18:00:00+05:30,2016.0,2026.0,11.0
1,301,1166,"Vikas Sadan, Gurugram - HSPCB",2016-03-25 13:30:00+05:30,2022-10-31 07:30:00+05:30,2016.0,2022.0,7.0
2,5509,14521,"Anand Vihar, Delhi - DPCC",2018-03-09 09:00:00+05:30,2022-10-31 06:00:00+05:30,2018.0,2022.0,5.0
3,5570,14717,"Aya Nagar, New Delhi - IMD",2018-03-09 11:00:00+05:30,2022-10-16 19:15:00+05:30,2018.0,2022.0,5.0
4,5626,14985,"DTU, New Delhi - CPCB",2018-03-09 11:00:00+05:30,2022-10-31 07:15:00+05:30,2018.0,2022.0,5.0
...,...,...,...,...,...,...,...,...
142,3409496,15646335,"New Moti Bagh, Delhi - MHUA",2026-03-10 16:00:00+05:30,2026-06-13 14:45:00+05:30,2026.0,2026.0,1.0
143,6299494,15888918,"Prashant Garden, Khora - UPPCB",2026-04-07 18:15:00+05:30,2026-09-11 16:00:00+05:30,2026.0,2026.0,1.0
144,6299678,15890341,"Ved Vihar-Loni, Ghaziabad - UPPCB",2026-04-07 22:30:00+05:30,2026-09-11 16:00:00+05:30,2026.0,2026.0,1.0
145,15,30,IGI Airport,NaT,NaT,NaN,NaN,NaN


In [38]:
two_year_sensors = coverage_table[
    coverage_table["coverage_years"] >= 2
]

print(
    "Sensors with at least two years of coverage:",
    len(two_year_sensors)
)

print(
    "Unique locations with at least two years of coverage:",
    two_year_sensors["location_id"].nunique()
)

Sensors with at least two years of coverage: 115
Unique locations with at least two years of coverage: 71


In [39]:
candidate_stations = coverage_table[
    coverage_table["coverage_years"] >= 2
].copy()

candidate_stations = candidate_stations.sort_values(
    by=["coverage_years", "last_reading"],
    ascending=[False, False]
)

display(candidate_stations.reset_index(drop=True))

,location_id,sensor_id,name,first_reading,last_reading,first_year,last_year,coverage_years
0,8118,23534,New Delhi,2016-11-10 00:30:00+05:30,2026-09-15 18:00:00+05:30,2016.0,2026.0,11.0
1,301,1166,"Vikas Sadan, Gurugram - HSPCB",2016-03-25 13:30:00+05:30,2022-10-31 07:30:00+05:30,2016.0,2022.0,7.0
2,5610,14860,"North Campus, DU, Delhi - IMD",2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,2018.0,2022.0,5.0
3,5617,14930,"Sector- 16A, Faridabad - HSPCB",2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,2018.0,2022.0,5.0
4,5622,14922,"NSIT Dwarka, Delhi - CPCB",2018-03-09 11:30:00+05:30,2022-10-31 07:30:00+05:30,2018.0,2022.0,5.0
...,...,...,...,...,...,...,...,...
110,6119272,14551755,"Sector - 5, Vasundhara, Ghaziabad, Uttar Pradesh",2025-11-04 20:30:00+05:30,2026-03-26 10:30:00+05:30,2025.0,2026.0,2.0
111,2587,13900,"Sector16A, Faridabad - HSPCB",2017-02-19 02:55:00+05:30,2018-02-22 08:55:00+05:30,2017.0,2018.0,2.0
112,2503,13860,"Shadipur, Delhi - CPCB",2017-02-19 00:00:00+05:30,2018-02-22 08:15:00+05:30,2017.0,2018.0,2.0
113,431,13868,"IHBAS, Delhi - CPCB",2017-02-19 02:15:00+05:30,2018-02-22 08:15:00+05:30,2017.0,2018.0,2.0
